# 14 — Final Panel Merge

Joins the monadic panel with network centrality measures to produce the single modelling-ready dataset.

**Inputs:**
- `../data/merged/panel_monadic_1992_2024.csv`
- `../data/merged/network_measures_1992_2024.csv`

**Output:** `../data/merged/panel_final_1992_2024.csv`

In [1]:
import numpy as np
import pandas as pd

## Step 1 — Load and merge

In [2]:
monadic = pd.read_csv('../../data/merged/panel_monadic_1992_2024.csv')
network = pd.read_csv('../../data/merged/network_measures_1992_2024.csv')

print(f'Monadic rows before merge:  {len(monadic):,}')
print(f'Network rows before merge:  {len(network):,}')

Monadic rows before merge:  6,358
Network rows before merge:  6,526


In [3]:
panel = monadic.merge(network, on=['recipient_iso3', 'year'], how='inner')

print(f'Rows after inner join: {len(panel):,}')

if len(panel) != 6358:
    raise ValueError(
        f'Expected 6,358 rows after merge, got {len(panel):,}. '
        'Check for duplicate keys or coverage gaps before proceeding.'
    )

print('Row count matches expected 6,358 — OK')

Rows after inner join: 6,358
Row count matches expected 6,358 — OK


## Step 2 — Log transforms

In [4]:
panel['arms_tiv_total_log'] = np.log1p(panel['arms_tiv_total'])
panel['oda_total_log']      = np.log1p(panel['oda_total'])

print('Log transforms added:')
print(panel[['arms_tiv_total', 'arms_tiv_total_log', 'oda_total', 'oda_total_log']].describe().round(3))

Log transforms added:
       arms_tiv_total  arms_tiv_total_log  oda_total  oda_total_log
count        6358.000            6358.000   6358.000       6358.000
mean          131.946               2.210    292.837          3.735
std           376.328               2.422    793.252          2.457
min             0.000               0.000      0.000          0.000
25%             0.000               0.000      3.940          1.597
50%             3.000               1.386     72.385          4.296
75%            68.160               4.236    282.228          5.646
max          5271.930               8.570  22007.840          9.999


## Step 3 — Lag main predictors

Network measures from `network_measures_1992_2024.csv` are already lagged by 1 year (built in notebook 11).  
Monadic flow totals are not yet lagged — add 1-year lags here.

In [ ]:
panel = panel.sort_values(['recipient_iso3', 'year']).reset_index(drop=True)

for col in ['arms_tiv_total_log', 'oda_total_log', 'econ_neocol_score_total']:
    panel[f'{col}_lag1'] = panel.groupby('recipient_iso3')[col].shift(1)

# Print only the 3 flow lag columns just created here.
# Network _lag1 columns (20 total) were already in the merge from network_measures_1992_2024.csv.
print('Flow lag columns added this step:',
      ['arms_tiv_total_log_lag1', 'oda_total_log_lag1', 'econ_neocol_score_total_lag1'])

## Step 4 — Validate

In [6]:
print(f'Row count: {len(panel):,}  (expected 6,358)')
assert len(panel) == 6358, f'Row count changed to {len(panel):,}'

Row count: 6,358  (expected 6,358)


In [7]:
# Missing % per column
missing_pct = (panel.isnull().sum() / len(panel) * 100).round(2)
missing_df = missing_pct[missing_pct > 0].sort_values(ascending=False)

print('Missing % (columns with any missing):')
print(missing_df.to_string())

# Lagged flow variables should have ~3.3% missing (first year per country)
flow_lag_cols = ['arms_tiv_total_log_lag1', 'oda_total_log_lag1', 'econ_neocol_score_total_lag1']
for col in flow_lag_cols:
    pct = missing_pct[col]
    flag = ' *** UNEXPECTED — INVESTIGATE' if pct > 5 else ''
    print(f'  {col}: {pct}% missing{flag}')

Missing % (columns with any missing):
gdp_per_capita                               6.35
gdp_per_capita_log                           6.35
population                                   4.22
population_log                               4.22
armed_conflict                               3.70
conflict_intensity                           3.70
oda_total_log_lag1                           3.35
arms_tiv_total_log_lag1                      3.35
econ_neocol_score_total_lag1                 3.35
arms_tiv_in_concentration_lag1               3.27
arms_tiv_pagerank_lag1                       3.27
arms_tiv_in_strength_lag1                    3.27
arms_tiv_out_strength_lag1                   3.27
econ_neocol_score_pagerank_lag1              3.27
econ_neocol_score_in_concentration_lag1      3.27
econ_neocol_score_dependency_balance_lag1    3.27
econ_neocol_score_out_strength_lag1          3.27
econ_neocol_score_in_strength_lag1           3.27
colonial_tie_pagerank_lag1                   3.27
colonial_tie

In [8]:
# Column inventory grouped by role
identifiers = ['recipient_iso3', 'year']

raw_predictors = [
    'arms_tiv_total', 'oda_total', 'econ_neocol_score_total', 'colonial_tie_flag'
]

log_transforms = ['arms_tiv_total_log', 'oda_total_log', 'gdp_per_capita_log', 'population_log']

lagged_predictors = [
    'arms_tiv_total_log_lag1', 'oda_total_log_lag1', 'econ_neocol_score_total_lag1'
]

# Network measures — already lagged (built in nb 11)
network_measures = [
    'arms_tiv_in_strength_lag1',        'arms_tiv_pagerank_lag1',
    'bilateral_oda_in_strength_lag1',   'bilateral_oda_pagerank_lag1',
    'econ_neocol_score_in_strength_lag1','econ_neocol_score_pagerank_lag1',
    'colonial_tie_in_strength_lag1',    'colonial_tie_pagerank_lag1',
]

# Additional network measures present (not in baseline feature lists but retained)
network_extra = [
    'arms_tiv_out_strength_lag1',              'arms_tiv_dependency_balance_lag1',
    'arms_tiv_in_concentration_lag1',
    'bilateral_oda_out_strength_lag1',         'bilateral_oda_dependency_balance_lag1',
    'bilateral_oda_in_concentration_lag1',
    'econ_neocol_score_out_strength_lag1',     'econ_neocol_score_dependency_balance_lag1',
    'econ_neocol_score_in_concentration_lag1',
    'colonial_tie_out_strength_lag1',          'colonial_tie_dependency_balance_lag1',
    'colonial_tie_in_concentration_lag1',
]

controls = ['gdp_per_capita', 'population', 'armed_conflict', 'conflict_intensity']

target = ['journalist_killings']

print('=== Column inventory ===')
for group, cols in [
    ('IDENTIFIERS',         identifiers),
    ('RAW PREDICTORS',      raw_predictors),
    ('LOG TRANSFORMS',      log_transforms),
    ('LAGGED PREDICTORS',   lagged_predictors),
    ('NETWORK (core)',       network_measures),
    ('NETWORK (extra)',      network_extra),
    ('CONTROLS',            controls),
    ('TARGET',              target),
]:
    print(f'\n{group} ({len(cols)})')
    for c in cols:
        in_panel = '✓' if c in panel.columns else '✗ MISSING'
        print(f'  {in_panel}  {c}')

=== Column inventory ===

IDENTIFIERS (2)
  ✓  recipient_iso3
  ✓  year

RAW PREDICTORS (4)
  ✓  arms_tiv_total
  ✓  oda_total
  ✓  econ_neocol_score_total
  ✓  colonial_tie_flag

LOG TRANSFORMS (4)
  ✓  arms_tiv_total_log
  ✓  oda_total_log
  ✓  gdp_per_capita_log
  ✓  population_log

LAGGED PREDICTORS (3)
  ✓  arms_tiv_total_log_lag1
  ✓  oda_total_log_lag1
  ✓  econ_neocol_score_total_lag1

NETWORK (core) (8)
  ✓  arms_tiv_in_strength_lag1
  ✓  arms_tiv_pagerank_lag1
  ✓  bilateral_oda_in_strength_lag1
  ✓  bilateral_oda_pagerank_lag1
  ✓  econ_neocol_score_in_strength_lag1
  ✓  econ_neocol_score_pagerank_lag1
  ✓  colonial_tie_in_strength_lag1
  ✓  colonial_tie_pagerank_lag1

NETWORK (extra) (12)
  ✓  arms_tiv_out_strength_lag1
  ✓  arms_tiv_dependency_balance_lag1
  ✓  arms_tiv_in_concentration_lag1
  ✓  bilateral_oda_out_strength_lag1
  ✓  bilateral_oda_dependency_balance_lag1
  ✓  bilateral_oda_in_concentration_lag1
  ✓  econ_neocol_score_out_strength_lag1
  ✓  econ_neocol_score

In [9]:
# Confirm key countries present
key_countries = ['IRQ', 'SYR', 'PHL', 'MEX', 'PAK']
present = panel['recipient_iso3'].unique()

for iso in key_countries:
    status = 'present' if iso in present else 'MISSING'
    print(f'  {iso}: {status}')

  IRQ: present
  SYR: present
  PHL: present
  MEX: present
  PAK: present


## Step 5 — Feature lists for modelling

Network column names come from `network_measures_1992_2024.csv` (already lagged with `_lag1` suffix).  
These lists are imported directly into the modelling notebook.

In [10]:
baseline_features = [
    'arms_tiv_total_log_lag1',
    'oda_total_log_lag1',
    'econ_neocol_score_total_lag1',
    'colonial_tie_flag',
    'gdp_per_capita_log',
    'population_log',
    'armed_conflict',
    'conflict_intensity',
]

# Network measures are already lagged — use as-is
network_features = baseline_features + [
    'arms_tiv_in_strength_lag1',         'arms_tiv_pagerank_lag1',
    'bilateral_oda_in_strength_lag1',    'bilateral_oda_pagerank_lag1',
    'econ_neocol_score_in_strength_lag1','econ_neocol_score_pagerank_lag1',
    'colonial_tie_in_strength_lag1',     'colonial_tie_pagerank_lag1',
]

print('baseline_features:', baseline_features)
print()
print('network_features:', network_features)
print()
print(f'Baseline: {len(baseline_features)} features')
print(f'Network-augmented: {len(network_features)} features')

# Sanity check — all features present in panel
missing_feats = [f for f in network_features if f not in panel.columns]
if missing_feats:
    print(f'\nWARNING — features not in panel: {missing_feats}')
else:
    print('\nAll features confirmed present in panel.')

baseline_features: ['arms_tiv_total_log_lag1', 'oda_total_log_lag1', 'econ_neocol_score_total_lag1', 'colonial_tie_flag', 'gdp_per_capita_log', 'population_log', 'armed_conflict', 'conflict_intensity']

network_features: ['arms_tiv_total_log_lag1', 'oda_total_log_lag1', 'econ_neocol_score_total_lag1', 'colonial_tie_flag', 'gdp_per_capita_log', 'population_log', 'armed_conflict', 'conflict_intensity', 'arms_tiv_in_strength_lag1', 'arms_tiv_pagerank_lag1', 'bilateral_oda_in_strength_lag1', 'bilateral_oda_pagerank_lag1', 'econ_neocol_score_in_strength_lag1', 'econ_neocol_score_pagerank_lag1', 'colonial_tie_in_strength_lag1', 'colonial_tie_pagerank_lag1']

Baseline: 8 features
Network-augmented: 16 features

All features confirmed present in panel.


## Step 6 — Save outputs

In [11]:
panel.to_csv('../../data/merged/panel_final_1992_2024.csv', index=False)

print(f'Saved: ../data/merged/panel_final_1992_2024.csv')
print(f'Shape: {panel.shape}')

Saved: ../data/merged/panel_final_1992_2024.csv
Shape: (6358, 38)


In [12]:
# Column inventory table for outputs/
col_records = []

role_map = {
    **{c: 'identifier'  for c in identifiers},
    **{c: 'predictor'   for c in raw_predictors},
    # log transforms of predictors stay predictor; GDP and population logs are controls
    **{c: 'predictor'   for c in log_transforms if c not in ('gdp_per_capita_log', 'population_log')},
    'gdp_per_capita_log': 'control',
    'population_log':     'control',
    **{c: 'predictor'   for c in lagged_predictors},
    **{c: 'network'     for c in network_measures},
    **{c: 'network'     for c in network_extra},
    **{c: 'control'     for c in controls},
    **{c: 'target'      for c in target},
}

for col in panel.columns:
    col_records.append({
        'variable_name': col,
        'type':          role_map.get(col, 'other'),
        'logged':        'yes' if '_log' in col else 'no',
        'lagged':        'yes' if '_lag1' in col else 'no',
    })

col_inventory = pd.DataFrame(col_records)
col_inventory.to_csv('../../outputs/data&methods/panel_final_columns.csv', index=False)

print('Saved: ../outputs/data&methods/panel_final_columns.csv')
print()
print(col_inventory.to_string(index=False))

Saved: ../outputs/data&methods/panel_final_columns.csv

                            variable_name       type logged lagged
                           recipient_iso3 identifier     no     no
                                     year identifier     no     no
                           arms_tiv_total  predictor     no     no
                                oda_total  predictor     no     no
                  econ_neocol_score_total  predictor     no     no
                        colonial_tie_flag  predictor     no     no
                      journalist_killings     target     no     no
                           gdp_per_capita    control     no     no
                       gdp_per_capita_log    control    yes     no
                               population    control     no     no
                           population_log    control    yes     no
                           armed_conflict    control     no     no
                       conflict_intensity    control     no     no
      